# Clase 05 — Detección no supervisada de anomalías en cardiotocografías

**Dataset:** [Cardiotocography](https://archive.ics.uci.edu/dataset/193/cardiotocography) (UCI) — 2126 cardiotocogramas (CTG) fetales con 21 variables medidas automáticamente (frecuencia cardíaca basal, aceleraciones, desaceleraciones, variabilidad, histograma de la frecuencia cardíaca). Tres obstetras clasificaron cada CTG según el estado fetal (`NSP`): normal, sospechoso o patológico.

Se busca un método no supervisado que detecte los CTG no normales (`NSP` > 1) mejor que los vistos en clase (`Cardiotocography outliers.ipynb`), en particular con menos de 400 anomalías no detectadas (FN). La etiqueta no se usa para ajustar los métodos: sólo para evaluarlos.

In [1]:
import sys
from pathlib import Path

# Permite importar el paquete `src` de la clase (la carpeta que contiene este notebook).
CLASE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
if str(CLASE_ROOT) not in sys.path:
    sys.path.insert(0, str(CLASE_ROOT))

import pandas as pd

from src.config import CLINICAL_FEATURES, COMPARISON_CSV, CONTAMINATION, RISK_DIRECTION
from src.data import load_ctg, log_standardize, robust_scale
from src.detectors import (
    center_distance,
    class_methods,
    directional_distance,
    flag_top,
    isolation_score,
    kmeans_distance,
    knn_distance,
    lof_score,
    pyod_score,
)
from src.evaluation import evaluate

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

## 1. Carga del dataset

`load_ctg` descarga el CSV de UCI la primera vez y lo guarda en `data/raw/cardiotocography.csv`. Devuelve las 21 variables y la etiqueta de anomalía: 1 si el CTG es sospechoso o patológico.

In [2]:
X, y = load_ctg()

print(f"Registros: {len(X)}, variables: {X.shape[1]}")
print(f"Anomalías (NSP > 1): {y.sum()} ({y.mean():.1%})")

Registros: 2126, variables: 21
Anomalías (NSP > 1): 471 (22.2%)


Las anomalías no son raras: son el 22% de los registros. Esto condiciona todo el análisis. Si un método marca sólo el 1% o el 5% de los datos, como en el notebook de clase, puede detectar a lo sumo 21 o 107 anomalías, así que sus FN no pueden bajar de 450 o 364, respectivamente.

## 2. Preprocesamiento

El notebook de clase escala con `RobustScaler`: (x − mediana) / IQR. Varias variables tienen la mediana y el primer cuartil en 0 y una cola larga a la derecha, por lo que quedan con valores enormes y dominan cualquier distancia. Se propone aplicar log(1 + x) a las variables sesgadas antes de estandarizar con z-score (`log_standardize`). Las tasas por segundo se pasan antes a eventos cada 1000 s, para que el logaritmo tenga efecto.

In [3]:
escalas = pd.DataFrame({
    "RobustScaler (clase)": robust_scale(X).max(),
    "log + z-score (propuesto)": log_standardize(X).max(),
})
escalas.loc[["FM", "Variance", "Nzeros", "ALTV", "MSTV", "DS"]].style.format("{:.3g}").set_caption(
    "Tabla 1. Valor máximo de algunas variables sesgadas según el escalado"
)

,RobustScaler (clase),log + z-score (propuesto)
FM,160,4.31
Variance,11.9,2.61
Nzeros,10,5.94
ALTV,8.27,2.24
MSTV,5.8,3.8
DS,0.001,17.4


`RobustScaler` falla en los dos sentidos:

- Los movimientos fetales (`FM`) llegan a 160: una sola variable define la distancia de los registros extremos.
- Las desaceleraciones severas (`DS`) quedan con un máximo de 0,001. Su IQR es 0, así que `RobustScaler` las deja en su valor crudo (eventos por segundo) y su aporte a la distancia desaparece. Y es una de las variables clínicas más importantes: de los 7 CTG con desaceleraciones severas, 6 son no normales.

Con log + z-score, `FM` baja a 4,3 y `DS` sube a 17,4: los pocos CTG con desaceleraciones severas pasan a pesar en la distancia, como corresponde.

## 3. Métodos del notebook de clase

Se reproducen los métodos de `Cardiotocography outliers.ipynb`, con su mismo escalado y sus mismos umbrales. Los resultados coinciden con los del notebook.

In [4]:
Z_clase = robust_scale(X)

filas = []
for metodo, (prediccion, puntaje, umbral) in class_methods(Z_clase).items():
    filas.append({"Método": metodo, "Umbral": umbral, **evaluate(y, prediccion, puntaje)})
tabla_clase = pd.DataFrame(filas).set_index("Método")

tabla_clase.style.format(
    {"Precisión": "{:.1%}", "Recall": "{:.1%}", "ROC-AUC": "{:.3f}"}, na_rep="—"
).set_caption("Tabla 2. Métodos del notebook de clase, con sus umbrales originales")

,Umbral,Marcados,TP,FP,FN,Precisión,Recall,ROC-AUC
Método,,,,,,,,
Z-score de la distancia al centro,z > 3,30,15,15,456,50.0%,3.2%,0.721
Tukey fences sobre la distancia al centro,Q3 + 2·IQR,158,63,95,408,39.9%,13.4%,0.721
"k-means (k = 10), distancia al centroide",percentil 99,22,14,8,457,63.6%,3.0%,0.641
"k-NN (k = 10), distancia media",percentil 99,22,12,10,459,54.5%,2.5%,0.582
"DBSCAN (eps = 5, min_samples = 5)",puntos de ruido,30,19,11,452,63.3%,4.0%,—
LOF (k = 10),contaminación 5%,107,40,67,431,37.4%,8.5%,0.524
Isolation Forest,contaminación 5%,107,68,39,403,63.6%,14.4%,0.727


Ningún método baja de 400 FN; el mejor es Isolation Forest, con 403. Pero la tabla no compara a los métodos en igualdad de condiciones: cada uno marca una cantidad distinta de registros (de 22 a 158), y con menos marcados siempre hay menos FP y más FN.

## 4. Métodos propuestos

1. **Distancia euclídea al centro con log + z-score**: el mismo método de la clase, con el preprocesamiento de la sección 2.
2. **La misma distancia, sólo con variables clínicas**: las 8 variables que los obstetras usan para evaluar el bienestar fetal (frecuencia basal, aceleraciones, desaceleraciones y variabilidad). Se eligen por criterio clínico, no por su relación con la etiqueta.
3. **Distancia direccional**: la distancia euclídea castiga cualquier desvío del centro, también los que van "hacia lo sano", como tener más variabilidad que el promedio. Esta versión sólo cuenta los desvíos en la dirección de riesgo de cada variable, también definida por criterio clínico (Tabla 3).

### Métodos de la librería PyOD

Se agregan siete detectores de [PyOD](https://pyod.readthedocs.io), con sus parámetros por defecto:

| Método | Idea | Escalado |
|---|---|---|
| **ECOD** (Li et al., IEEE TKDE 2022) | Distribución acumulada empírica de cada variable: suma el −log de la probabilidad de cola de cada registro, eligiendo la cola por la asimetría de la variable | sin escalar (trabaja con rangos) |
| **COPOD** (Li et al., ICDM 2020) | La misma idea que ECOD, con una cópula empírica | sin escalar (trabaja con rangos) |
| **HBOS** | Un histograma por variable; asume independencia entre variables. Es muy rápido | log + z-score |
| **ABOD** | Varianza de los ángulos hacia los vecinos: un punto aislado ve a los demás en ángulos parecidos. Pensado para alta dimensión | log + z-score |
| **SOD** | Desvío en el subespacio de los ejes donde varían poco sus vecinos | log + z-score |
| **COF** | Densidad medida a lo largo del camino que lo conecta con sus vecinos | log + z-score |
| **LODA** | Ensamble de histogramas sobre proyecciones aleatorias | log + z-score |

ECOD se parece a la distancia direccional: mira cada variable por separado y sólo cuenta una cola. La diferencia es que elige la cola por la asimetría de cada variable, no por su significado clínico. Por eso entra con las 21 variables y también con las 8 clínicas; los demás, con las 21.

ABOD da un puntaje indefinido en los registros duplicados (con dos puntos idénticos, el ángulo entre ellos no existe). Como un registro con un duplicado exacto no está aislado, recibe el puntaje mínimo (`pyod_score`).

In [5]:
descripcion = {1: "valores altos", -1: "valores bajos", 0: "cualquier desvío"}
pd.Series({variable: descripcion[signo] for variable, signo in RISK_DIRECTION.items()}, name="se considera de riesgo") \
    .to_frame().style.set_caption("Tabla 3. Dirección de riesgo de cada variable clínica")

,se considera de riesgo
LB,cualquier desvío
AC,valores bajos
DL,valores altos
DS,valores altos
DP,valores altos
ASTV,valores altos
MSTV,valores bajos
ALTV,valores altos


## 5. Tabla comparativa con el mismo umbral

Para comparar en igualdad de condiciones, todos los métodos marcan como anomalía el mismo 5% de los registros con mayor puntaje (107 CTG), que es la contaminación que usan LOF e Isolation Forest en el notebook de clase. DBSCAN queda afuera porque no da un puntaje continuo.

El **ROC-AUC** no depende del umbral: mide qué tan bien ordena el puntaje a las anomalías por encima de los normales (0,5 es un orden al azar y 1, perfecto).

In [6]:
Z_log = log_standardize(X)
Z_clinicas = log_standardize(X[CLINICAL_FEATURES])

puntajes = {
    ("Clase", "Distancia euclídea al centro", "RobustScaler", "21"): center_distance(Z_clase),
    ("Clase", "k-means (k = 10), distancia al centroide", "RobustScaler", "21"): kmeans_distance(Z_clase),
    ("Clase", "k-NN (k = 10), distancia media", "RobustScaler", "21"): knn_distance(Z_clase),
    ("Clase", "LOF (k = 10)", "RobustScaler", "21"): lof_score(Z_clase),
    ("Clase", "Isolation Forest", "RobustScaler", "21"): isolation_score(Z_clase),
    ("Propuesto", "Distancia euclídea al centro", "log + z-score", "21"): center_distance(Z_log),
    ("Propuesto", "Distancia euclídea al centro", "log + z-score", "8 clínicas"): center_distance(Z_clinicas),
    ("Propuesto", "Distancia direccional (sólo desvíos de riesgo)", "log + z-score", "8 clínicas"): directional_distance(Z_clinicas),
    ("PyOD", "ECOD", "sin escalar", "21"): pyod_score("ECOD", X),
    ("PyOD", "ECOD", "sin escalar", "8 clínicas"): pyod_score("ECOD", X[CLINICAL_FEATURES]),
    ("PyOD", "COPOD", "sin escalar", "21"): pyod_score("COPOD", X),
    **{
        ("PyOD", metodo, "log + z-score", "21"): pyod_score(metodo, Z_log)
        for metodo in ["HBOS", "ABOD", "SOD", "COF", "LODA"]
    },
}

comparacion = pd.DataFrame(
    [evaluate(y, flag_top(puntaje, CONTAMINATION), puntaje) for puntaje in puntajes.values()],
    index=pd.MultiIndex.from_tuples(list(puntajes), names=["Origen", "Método", "Escalado", "Variables"]),
).sort_values("ROC-AUC", ascending=False)

COMPARISON_CSV.parent.mkdir(parents=True, exist_ok=True)
comparacion.to_csv(COMPARISON_CSV)

negrita = "font-weight: bold"
comparacion.style.format(
    {"Precisión": "{:.1%}", "Recall": "{:.1%}", "ROC-AUC": "{:.3f}"}
).highlight_max(subset=["TP", "Precisión", "Recall", "ROC-AUC"], props=negrita).highlight_min(
    subset=["FP", "FN"], props=negrita
).set_caption(f"Tabla 4. Comparación de métodos con el mismo umbral ({CONTAMINATION:.0%} de los registros)")

## 6. Conclusiones

- **El método más simple, bien preprocesado, supera a los de clase.** La distancia euclídea al centro con log + z-score pasa de un ROC-AUC de 0,72 a 0,80, y ya baja de 400 FN (394).
- **Elegir variables con criterio clínico** mejora tanto el orden (ROC-AUC 0,82) como el umbral: 90 TP, 17 FP y 381 FN.
- **La distancia direccional es el mejor método** (ROC-AUC 0,91): 96 TP, 11 FP y 375 FN, con una precisión del 90%. Frente al mejor método de clase (Isolation Forest con su umbral original: 68 TP, 39 FP y 403 FN) detecta más anomalías y tiene menos falsas alarmas a la vez.
- **ECOD ordena mejor que todos los métodos de clase** (ROC-AUC 0,78 con 21 variables y 0,80 con las 8 clínicas), sin ningún ajuste. Pero al 5% no baja de 400 FN (409 y 406). Tiene el mismo problema que la distancia simétrica: muchos de sus falsos positivos tienen más variabilidad que el promedio (`MSTV` alta, `ASTV` baja), un signo sano. `MSTV` es asimétrica a la derecha, así que ECOD mira su cola derecha, cuando el riesgo clínico es la variabilidad baja. Que la distancia direccional lo supere (0,91 contra 0,80 con las mismas variables) muestra que la dirección clínica aporta más que la forma de las distribuciones.
- **El resto de los métodos de PyOD no mejora a los de clase.** COPOD, LODA y HBOS ordenan algo mejor que el azar (ROC-AUC de 0,64 a 0,66), pero por debajo de Isolation Forest (0,73), y ninguno baja de 400 FN. COPOD queda muy por debajo de ECOD (0,66 contra 0,78), a pesar de la idea compartida.
- **Los métodos locales son los peores de la tabla:** k-NN, LOF, COF y SOD quedan con un ROC-AUC de 0,52 a 0,58, cerca del azar, y ABOD queda por debajo (0,47): ordena los CTG no normales como *menos* anómalos que los normales. Los CTG no normales no son puntos raros y aislados: son el 22% de los datos y forman sus propios grupos densos, así que localmente no parecen anómalos. Que ABOD esté pensado para alta dimensión tampoco ayuda: con 21 variables, el problema no es la dimensión sino que las anomalías no están aisladas.
- **El umbral pesa tanto como el método.** Con 107 registros marcados, el FN no puede bajar de 364. Si se marcara la proporción esperada de CTG no normales (22%), los FN bajarían mucho, a costa de más falsas alarmas.

**Limitación:** la elección de variables y de la dirección de riesgo sale del criterio clínico, no de las etiquetas. Pero haber comparado varios métodos contra la misma etiqueta hace que el mejor resultado sea algo optimista; para confirmarlo habría que validarlo en otro conjunto de CTG.